# Generator configuration

In [1]:
# Load the modules
import pandas as pd
import numpy as np
import sdv
import re
from sdv.single_table import CTGANSynthesizer
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.single_table import TVAESynthesizer
from sdv.metadata import Metadata
from synthcity.plugins import Plugins
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import torch
import random
import os.path
from os import listdir
from os.path import isfile, join
import faiss
import time
import sys
import platform
from sdv.evaluation.single_table import evaluate_quality
from scipy.stats import linregress
from matplotlib.ticker import ScalarFormatter
import re

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


#### 1 --- CTGAN

In [2]:
# ============================================================
# Functions
# ============================================================

def set_seed(seed=0):
    """
    Enable deterministic mode and set all random seeds.
    """

    print(f"DETERMINISTIC MODE is enabled (seed={seed})")

    # --------------------------------------------------------
    # Set global random seeds
    # --------------------------------------------------------
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    # --------------------------------------------------------
    # Deterministic CUDA operations
    # --------------------------------------------------------
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.use_deterministic_algorithms(True)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False




# ============================================================
# Final loss window analysis
# ============================================================
final_window_fraction = 0.30

def analyze_final_loss_window(loss_data, window_fraction=0.30):
    """
    Calculate final-window statistics for Generator and Discriminator losses.

    Statistics returned for each model:
        - slope

    The final window corresponds to the last fraction
    of the complete training trajectory.
    """
    
    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------
    required_columns = ["Epoch", "Generator Loss", "Discriminator Loss"]
    missing_columns = [col for col in required_columns if col not in loss_data.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    # --------------------------------------------------------
    # Validate window fraction
    # --------------------------------------------------------
    if not (0 < window_fraction <= 1):
        raise ValueError("window_fraction must be between 0 and 1.")

    # --------------------------------------------------------
    # Sort epochs
    # --------------------------------------------------------
    loss_data = loss_data.sort_values("Epoch").copy()
    n_epochs = len(loss_data)
    if n_epochs < 2:
        raise ValueError("At least two epochs are required.")

    # --------------------------------------------------------
    # Determine final window size
    # --------------------------------------------------------
    window_size = max(2, int(np.ceil(n_epochs * window_fraction)))
    window_size = min(window_size, n_epochs)

    # --------------------------------------------------------
    # Extract final window
    # --------------------------------------------------------
    final_window = loss_data.tail(window_size).copy()

    def _window_stats(series: pd.Series, epochs: pd.Series) -> dict:
        values = series.astype(float).to_numpy()
        regression = linregress(epochs, values)
        return {
            "slope": float(regression.slope),
        }

    # ========================================================
    # GENERATOR LOSS
    # ========================================================
    generator_stats = _window_stats(
        final_window["Generator Loss"], final_window["Epoch"]
    )

    # ========================================================
    # DISCRIMINATOR LOSS
    # ========================================================
    discriminator_stats = _window_stats(
        final_window["Discriminator Loss"], final_window["Epoch"]
    )

    # ========================================================
    # RETURN
    # ========================================================
    return {
        "window_size": window_size,
        "window_fraction": window_fraction,
        "epoch_range": (int(final_window["Epoch"].min()), int(final_window["Epoch"].max())),
        "G_slope": generator_stats["slope"],
        "D_slope": discriminator_stats["slope"],
    }


# ============================================================
# Application to both training regimes (300 / 1000 epochs)
# ============================================================
def summarize_stability(df: pd.DataFrame, window_fraction: float = final_window_fraction) -> pd.DataFrame:
    """Apply the stability analysis to each training regime in the dataset."""
    rows = []
    for n_epochs in sorted(df["training_epochs"].unique()):
        subset = df[df["training_epochs"] == n_epochs]
        stats = analyze_final_loss_window(subset, window_fraction=window_fraction)
        rows.append({
            "training_epochs": n_epochs,
            "window_size": stats["window_size"],
            "epoch_range": f"{stats['epoch_range'][0]}–{stats['epoch_range'][1]}",
            "G_slope": stats["G_slope"],
            "D_slope": stats["D_slope"],
        })
    return pd.DataFrame(rows)


##### PIMA dataset

In [3]:
# ============================================================
# Load PIMA dataset
# ============================================================
data_path = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data/PIMA_train.csv")

df = pd.read_csv(data_path,keep_default_na=False)

# ============================================================
# Detect metadata
# ============================================================

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name="pima"
)

# ============================================================
# CONFIGURATION
# ============================================================

output_dir = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data")
os.makedirs(output_dir, exist_ok=True)

# Training budgets
epoch_values = [300,500,600,700,1000]


# ============================================================
# STORAGE
# ============================================================

loss_results = []
stability_results = []
quality_results =[]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

for epochs in epoch_values:
    synthesizer= CTGANSynthesizer(
        metadata=metadata,
        epochs=epochs,
        batch_size=70,
        generator_dim=(256,256),      
        discriminator_dim=(256, 256),
        verbose=True,
        cuda=False
    )
    
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=len(df))
    # quality assessement
    quality_report = evaluate_quality(
        real_data=df,
        synthetic_data=synthetic_data,
        metadata=metadata
    )

    quality_score = (quality_report.get_score())
    quality_results.append(quality_score)

    # losses functions
    loss_df = (synthesizer.get_loss_values().copy())
    loss_df["training_epochs"] = epochs
    loss_results.append(loss_df)

pima_loss_df_all_ctgan = pd.concat(loss_results,ignore_index=True)
#pima_loss_df_all_ctgan.to_csv(os.path.join(output_dir,"pima_loss_results_ctgan.csv"),index=False)

print(quality_results)

if __name__ == "__main__":
    pima_summary_ctgan = summarize_stability(pima_loss_df_all_ctgan, window_fraction=final_window_fraction)

    pd.set_option("display.float_format", lambda x: f"{x:.5f}")
    print(f"\nStability analysis — final window = {int(final_window_fraction * 100)}% des epochs\n")
    print(pima_summary_ctgan.to_string(index=False))
    #pima_summary_ctgan.to_csv(os.path.join(output_dir,"pima_slope_results_ctgan.csv"),index=False)


Gen. (-2.27) | Discrim. (0.08): 100%|█████████| 300/300 [00:26<00:00, 11.19it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 813.09it/s]|
Column Shapes Score: 82.25%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 261.65it/s]|
Column Pair Trends Score: 77.19%

Overall Score (Average): 79.72%



Gen. (-2.63) | Discrim. (-0.60): 100%|████████| 500/500 [00:48<00:00, 10.29it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 818.33it/s]|
Column Shapes Score: 86.86%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 259.10it/s]|
Column Pair Trends Score: 77.73%

Overall Score (Average): 82.29%



Gen. (-2.52) | Discrim. (0.19): 100%|█████████| 600/600 [00:56<00:00, 10.65it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 998.09it/s]|
Column Shapes Score: 88.58%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 296.69it/s]|
Column Pair Trends Score: 80.06%

Overall Score (Average): 84.32%



Gen. (-2.51) | Discrim. (0.27): 100%|█████████| 700/700 [01:07<00:00, 10.40it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 815.89it/s]|
Column Shapes Score: 82.74%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 277.92it/s]|
Column Pair Trends Score: 74.97%

Overall Score (Average): 78.86%



Gen. (-1.34) | Discrim. (-0.51): 100%|██████| 1000/1000 [01:38<00:00, 10.10it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 960.70it/s]|
Column Shapes Score: 85.62%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 329.21it/s]|
Column Pair Trends Score: 76.44%

Overall Score (Average): 81.03%

[0.7971783229273295, 0.8229360836817512, 0.8431685500175063, 0.7885722596622403, 0.8103200950828116]

Stability analysis — final window = 30% des epochs

 training_epochs  window_size epoch_range  G_slope  D_slope
             300           90     210–299  0.00530  0.00278
             500          150     350–499  0.00083  0.00015
             600          180     420–599 -0.00052  0.00029
             700          210     490–699  0.00202 -0.00028
            1000          300     700–999  0.00194  0.00013


#### AIDS dataset

In [221]:
# ============================================================
# Load AIDS dataset
# ============================================================
data_path = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data/AIDS_train.csv")

df = pd.read_csv(data_path,keep_default_na=False)

# ============================================================
# Detect metadata
# ============================================================

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name='hotel_guests'
)


# ============================================================
# CONFIGURATION
# ============================================================

output_dir = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data")
os.makedirs(output_dir, exist_ok=True)

# Training budgets
epoch_values = [300,500,600,1000,1200,1500,2000]


# ============================================================
# STORAGE
# ============================================================

loss_results = []
stability_results = []
quality_results =[]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

for epochs in epoch_values:
    synthesizer= CTGANSynthesizer(
        metadata=metadata,
        epochs=epochs,
        batch_size=200,
        generator_dim=(128,256),     
        discriminator_dim=(256, 128),
        verbose=True,
        cuda=False
    )
    
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=len(df))
    # quality assessement
    quality_report = evaluate_quality(
        real_data=df,
        synthetic_data=synthetic_data,
        metadata=metadata
    )

    quality_score = (quality_report.get_score())
    quality_results.append(quality_score)

    # losses functions
    loss_df = (synthesizer.get_loss_values().copy())
    loss_df["training_epochs"] = epochs
    loss_results.append(loss_df)

aids_loss_df_all_ctgan = pd.concat(loss_results,ignore_index=True)
aids_loss_df_all_ctgan.to_csv(os.path.join(output_dir,"aids_loss_results_ctgan.csv"),index=False)

print(quality_results)

if __name__ == "__main__":
    aids_summary_ctgan = summarize_stability(aids_loss_df_all_ctgan, window_fraction=final_window_fraction)

    pd.set_option("display.float_format", lambda x: f"{x:.5f}")
    print(f"\nStability analysis — final window = {int(final_window_fraction * 100)}% des epochs\n")
    print(aids_summary_ctgan.to_string(index=False))
    aids_summary_ctgan.to_csv(os.path.join(output_dir,"aids_slope_results_ctgan.csv"),index=False)

Gen. (-3.01) | Discrim. (-0.05): 100%|████████| 300/300 [01:00<00:00,  4.98it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1236.75it/s]|
Column Shapes Score: 86.58%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 387.34it/s]|
Column Pair Trends Score: 74.6%

Overall Score (Average): 80.59%



Gen. (-2.72) | Discrim. (-0.58): 100%|████████| 500/500 [01:25<00:00,  5.85it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1104.16it/s]|
Column Shapes Score: 91.81%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 369.19it/s]|
Column Pair Trends Score: 86.41%

Overall Score (Average): 89.11%



Gen. (-3.26) | Discrim. (-0.35): 100%|████████| 600/600 [01:51<00:00,  5.40it/s]


...

Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1081.01it/s]|
Column Shapes Score: 93.23%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 359.18it/s]|
Column Pair Trends Score: 87.68%

Overall Score (Average): 90.45%



Gen. (-2.01) | Discrim. (-0.20): 100%|██████| 1500/1500 [05:03<00:00,  4.94it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 20/20 [00:00<00:00, 907.37it/s]|
Column Shapes Score: 93.58%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 284.15it/s]|
Column Pair Trends Score: 89.04%

Overall Score (Average): 91.31%



Gen. (-1.14) | Discrim. (-0.09): 100%|██████| 2000/2000 [06:50<00:00,  4.87it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1153.96it/s]|
Column Shapes Score: 91.89%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 411.34it/s]|
Column Pair Trends Score: 87.43%

Overall Score (Average): 89.66%

[0.805906256958361, 0.8910988643954576, 0.9029113783121799, 0.8898630594522379, 0.9045368514807393, 0.913098419060343, 0.8966210198174127]

Stability analysis — final window = 30% des epochs

 training_epochs  window_size epoch_range  G_slope  D_slope
             300           90     210–299  0.00025 -0.00077
             500          150     350–499  0.00528  0.00090
             600          180     420–599  0.00017  0.00027
            1000          300     700–999 -0.00033  0.00000
            1200          360    840–1199  0.00132 -0.00045
            1500          450   1050–1499  0.00120 -0.00008
            2000          600   1400–1999  0.00049 -0.00008


#### REIN dataset

In [81]:
# ============================================================
# Load REIN dataset
# ============================================================
data_path = (
    "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data/REIN_train.csv"
)

df = pd.read_csv(data_path,keep_default_na=False)

# Values that should actually be treated as missing
missing_values = ["NA", "N/A", ""]

df = df.replace(missing_values, pd.NA)

continuous_vars = [
    "Age",
    "Serum_albumin_level",
    "Body_mass_index",
    "eGFR",
    "times"
]

# Convert continuous variables to numeric
for col in continuous_vars:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# Detect metadata
# ============================================================

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name='hotel_guests'
)


# ============================================================
# CONFIGURATION
# ============================================================

output_dir = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data")
os.makedirs(output_dir, exist_ok=True)

# Training budgets
epoch_values = [300,500,800,1000,1200]


# ============================================================
# STORAGE
# ============================================================

loss_results = []
stability_results = []
quality_results =[]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

for epochs in epoch_values:
    synthesizer= CTGANSynthesizer(
        metadata=metadata,
        epochs=epochs,
        batch_size=3000,
        generator_dim=(128, 128),      
        discriminator_dim=(256, 256),
        verbose=True,
        cuda=False
    )
    
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=len(df))
    # quality assessement
    quality_report = evaluate_quality(
        real_data=df,
        synthetic_data=synthetic_data,
        metadata=metadata
    )

    quality_score = (quality_report.get_score())
    quality_results.append(quality_score)

    # losses functions
    loss_df = (synthesizer.get_loss_values().copy())
    loss_df["training_epochs"] = epochs
    loss_results.append(loss_df)

rein_loss_df_all_ctgan = pd.concat(loss_results,ignore_index=True)
rein_loss_df_all_ctgan.to_csv(os.path.join(output_dir,"rein_loss_results_ctgan.csv"),index=False)

print(quality_results)

if __name__ == "__main__":
    rein_summary_ctgan = summarize_stability(rein_loss_df_all_ctgan, window_fraction=final_window_fraction)

    pd.set_option("display.float_format", lambda x: f"{x:.5f}")
    print(f"\nStability analysis — final window = {int(final_window_fraction * 100)}% des epochs\n")
    print(rein_summary_ctgan.to_string(index=False))
    rein_summary_ctgan.to_csv(os.path.join(output_dir,"rein_slope_results_ctgan.csv"),index=False)

Gen. (-0.74) | Discrim. (0.01): 100%|█████████| 300/300 [04:03<00:00,  1.23it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 129.01it/s]|
Column Shapes Score: 93.41%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:00<00:00, 185.63it/s]|
Column Pair Trends Score: 92.99%

Overall Score (Average): 93.2%



Gen. (-1.09) | Discrim. (-0.18): 100%|████████| 500/500 [06:50<00:00,  1.22it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████████| 19/19 [00:00<00:00, 93.25it/s]|
Column Shapes Score: 95.37%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:01<00:00, 146.06it/s]|
Column Pair Trends Score: 94.07%

Overall Score (Average): 94.72%



Gen. (-0.88) | Discrim. (-0.08): 100%|████████| 800/800 [13:38<00:00,  1.02s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 111.03it/s]|
Column Shapes Score: 95.2%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:01<00:00, 169.58it/s]|
Column Pair Trends Score: 93.65%

Overall Score (Average): 94.42%



Gen. (-0.63) | Discrim. (-0.11): 100%|██████| 1000/1000 [16:28<00:00,  1.01it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 128.24it/s]|
Column Shapes Score: 94.27%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:00<00:00, 189.89it/s]|
Column Pair Trends Score: 94.11%

Overall Score (Average): 94.19%



Gen. (-1.35) | Discrim. (0.02): 100%|███████| 1200/1200 [21:00<00:00,  1.05s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 100.43it/s]|
Column Shapes Score: 96.72%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:01<00:00, 150.79it/s]|
Column Pair Trends Score: 95.0%

Overall Score (Average): 95.86%

[0.93197364409907, 0.9472258025411053, 0.9442459670159036, 0.9419315359835418, 0.9586000745181723]

Stability analysis — final window = 30% des epochs

 training_epochs  window_size epoch_range  G_slope  D_slope
             300           90     210–299 -0.00032  0.00088
             500          150     350–499 -0.00024 -0.00024
             800          240     560–799  0.00111 -0.00005
            1000          300     700–999 -0.00084  0.00006
            1200          360    840–1199 -0.00050 -0.00005


### figures

In [16]:
"""
CTGAN training losses (PIMA / AIDS / REIN) - npj-style figure.
Row 1: generator loss; Row 2: discriminator loss.
Outputs: PNG (600 dpi) and PDF (vector, embedded TrueType fonts).
"""
import os, re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---------------- paths (edit if needed) ----------------
DATA_DIR = "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data"
OUT_DIR= "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/figures"
BASENAME = "ctgan_training_losses_pima_aids_rein"
EPOCHS_CONFIG = {"PIMA": 600, "AIDS": 600, "REIN": 500}
os.makedirs(OUT_DIR, exist_ok=True)


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "mathtext.fontset": "custom",
    "mathtext.rm": "sans", "mathtext.it": "sans:italic", "mathtext.bf": "sans:bold",
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.4,
    "axes.axisbelow": True,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})

# Okabe-Ito (colour-blind safe); linestyles also differ (redundant coding)
C_G, C_D = "#0072B2", "#D55E00"        # blue, vermillion
C_FIT = "#000000"                      # black fit lines: max contrast
C_WIN = "#E69F00"                      # orange window shading (distinct from G/D)

def parse_range(s):
    a, b = re.split(r"[\u2013\-]", str(s))
    return int(a), int(b)

def prepare(name, n_ep):
    k = name.lower()
    df = pd.read_csv(os.path.join(DATA_DIR, f"{k}_loss_results_ctgan.csv"))
    sl = pd.read_csv(os.path.join(DATA_DIR, f"{k}_slope_results_ctgan.csv"))
    d = df[df.training_epochs == n_ep].sort_values("Epoch").reset_index(drop=True)
    r = sl[sl.training_epochs == n_ep]
    if d.empty or r.empty:
        raise ValueError(f"[{name}] no data for {n_ep} epochs")
    r = r.iloc[0]
    s, e = parse_range(r["epoch_range"])
    w = d[(d.Epoch >= s) & (d.Epoch <= e)]
    out = dict(name=name, n=n_ep, d=d, s=s, e=e, we=w.Epoch.to_numpy())
    for key, col, sc in (("g", "Generator Loss", "G_slope"), ("dd", "Discriminator Loss", "D_slope")):
        sp = float(r[sc]); v = w[col].to_numpy()
        out[key + "_slope"] = sp
        out[key + "_fit"] = sp * out["we"] + (v.mean() - sp * out["we"].mean())
    return out

def panel(ax, P, col, fit, slope, color, ls, label, title=None, ylabel=None, xlabel=False):
    d = P["d"]
    ax.axvspan(P["s"], P["e"], color=C_WIN, alpha=0.18, lw=0, zorder=0)
    ax.plot(d.Epoch, d[col], color=color, lw=0.6, ls=ls, alpha=0.95, zorder=2)
    ax.plot(P["we"], fit, color=C_FIT, lw=1.6, solid_capstyle="round", zorder=3)
    ax.axhline(0, color="0.4", lw=0.5, ls=":", zorder=1)
    ax.set_xlim(d.Epoch.min(), d.Epoch.max())
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.30 * (hi - lo))   # headroom for the slope annotation
    if title: ax.set_title(title, pad=7)
    if ylabel: ax.set_ylabel(ylabel)
    if xlabel: ax.set_xlabel("Epoch")
    ax.text(-0.02, 1.02, label, transform=ax.transAxes, fontsize=8, fontweight="bold",
            va="bottom", ha="right", clip_on=False)
    ax.text(0.97, 0.96, f"slope = {slope:+.1e} / epoch\n(epochs {P['s']}\u2013{P['e']})",
            transform=ax.transAxes, fontsize=5.8, va="top", ha="right", zorder=5,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", lw=0.4, alpha=0.9))

def main():
    data = [prepare(n, e) for n, e in EPOCHS_CONFIG.items()]
    fig, axes = plt.subplots(2, len(data), figsize=(7.09, 4.0), squeeze=False, sharex="col")
    letters = "abcdef"
    for j, P in enumerate(data):
        panel(axes[0, j], P, "Generator Loss", P["g_fit"], P["g_slope"], C_G, "-",
              letters[j], title=f"{P['name']} ({P['n']} epochs)",
              ylabel="Generator loss" if j == 0 else None)
        panel(axes[1, j], P, "Discriminator Loss", P["dd_fit"], P["dd_slope"], C_D, "-",
              letters[len(data) + j], ylabel="Discriminator loss" if j == 0 else None, xlabel=True)
    handles = [
        Line2D([0], [0], color=C_G, lw=1.2, label="Generator loss"),
        Line2D([0], [0], color=C_D, lw=1.2, label="Discriminator loss"),
        Line2D([0], [0], color=C_FIT, lw=1.6, label="Linear fit (final window)"),
        Patch(fc=C_WIN, alpha=0.3, ec="none", label="Final analysis window"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False,
               bbox_to_anchor=(0.5, 0.0), columnspacing=1.8, handlelength=2.0)
    fig.tight_layout(rect=[0, 0.04, 1, 1], w_pad=1.2, h_pad=0.8)
    for ext, kw in (("png", dict(dpi=600)), ("pdf", {})):
        p = os.path.join(OUT_DIR, f"{BASENAME}.{ext}")
        fig.savefig(p, bbox_inches="tight", facecolor="white", **kw)
        print("Saved:", p)

if __name__ == "__main__":
    main()

Saved: Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/figures/ctgan_training_losses_pima_aids_rein.png
Saved: Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/figures/ctgan_training_losses_pima_aids_rein.pdf


#### 2 --- TVAE

In [124]:
def analyze_final_loss_window(
    loss_data: pd.DataFrame,
    window_fraction: float = 0.30
) -> dict:
    """
    Calculate the slope of the TVAE loss over the final
    training window.

    The final window corresponds to the last fraction
    of the complete training trajectory.
    """

    required_columns = ["Epoch", "Loss"]

    missing_columns = [
        col
        for col in required_columns
        if col not in loss_data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    if not (0 < window_fraction <= 1):
        raise ValueError(
            "window_fraction must be between 0 and 1."
        )

    loss_data = (
        loss_data
        .sort_values("Epoch")
        .copy()
    )

    n_epochs = len(loss_data)

    if n_epochs < 2:
        raise ValueError(
            "At least two epochs are required."
        )

    window_size = max(
        2,
        int(np.ceil(n_epochs * window_fraction))
    )

    window_size = min(
        window_size,
        n_epochs
    )

    final_window = (
        loss_data
        .tail(window_size)
        .copy()
    )

    epochs = (
        final_window["Epoch"]
        .astype(float)
        .to_numpy()
    )

    values = (
        final_window["Loss"]
        .astype(float)
        .to_numpy()
    )

    regression = linregress(
        epochs,
        values
    )

    slope = float(
        regression.slope
    )

    return {
        "window_size": window_size,
        "window_fraction": window_fraction,
        "epoch_range": (
            int(final_window["Epoch"].min()),
            int(final_window["Epoch"].max())
        ),
        "Loss_slope": slope,
    }


def summarize_stability(
    df: pd.DataFrame,
    window_fraction: float = final_window_fraction
) -> pd.DataFrame:
    """
    Apply the stability analysis to each training regime
    available in the TVAE loss DataFrame.
    """

    rows = []

    for n_epochs in sorted(
        df["training_epochs"].unique()
    ):

        subset = df[
            df["training_epochs"] == n_epochs
        ].copy()

        stats = analyze_final_loss_window(
            subset,
            window_fraction=window_fraction
        )

        rows.append({
            "training_epochs": n_epochs,
            "window_size": stats["window_size"],
            "epoch_range": (
                f"{stats['epoch_range'][0]}"
                f"–"
                f"{stats['epoch_range'][1]}"
            ),
            "Loss_slope": stats["Loss_slope"],
        })

    return pd.DataFrame(rows)


#### PIMA dataset

In [125]:
# ============================================================
# Load PIMA dataset
# ============================================================
data_path = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data/PIMA_train.csv")

df = pd.read_csv(data_path,keep_default_na=False)

# ============================================================
# Detect metadata
# ============================================================

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name="pima"
)

# ============================================================
# CONFIGURATION
# ============================================================

output_dir = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data")
os.makedirs(output_dir, exist_ok=True)

# Training budgets
epoch_values = [300,500,600,700,1000,1200,1500]


# ============================================================
# STORAGE
# ============================================================

loss_results = []
stability_results = []
quality_results =[]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

for epochs in epoch_values:
    synthesizer = TVAESynthesizer(
    metadata,
    epochs=epochs,
    verbose=True
)
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=len(df))
    # quality assessement
    quality_report = evaluate_quality(
        real_data=df,
        synthetic_data=synthetic_data,
        metadata=metadata
    )

    quality_score = (quality_report.get_score())
    quality_results.append(quality_score)

    # losses functions
    loss_df = (synthesizer.get_loss_values().copy())
    loss_df["training_epochs"] = epochs
    loss_results.append(loss_df)

pima_loss_df_all_tvae = pd.concat(loss_results,ignore_index=True)
pima_loss_df_all_tvae.to_csv(os.path.join(output_dir,"pima_loss_results_tvae.csv"),index=False)

print(quality_results)

if __name__ == "__main__":
    pima_summary_tvae = summarize_stability(pima_loss_df_all_tvae, window_fraction=final_window_fraction)

    pd.set_option("display.float_format", lambda x: f"{x:.5f}")
    print(f"\nStability analysis — final window = {int(final_window_fraction * 100)}% des epochs\n")
    print(pima_summary_tvae.to_string(index=False))
    pima_summary_tvae.to_csv(os.path.join(output_dir,"pima_slope_results_tvae.csv"),index=False)


Loss: 1.070: 100%|████████████████████████████| 300/300 [00:19<00:00, 15.75it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 254.52it/s]|
Column Shapes Score: 84.61%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 160.28it/s]|
Column Pair Trends Score: 72.73%

Overall Score (Average): 78.67%



Loss: -1.141: 100%|███████████████████████████| 500/500 [00:26<00:00, 18.82it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 932.64it/s]|
Column Shapes Score: 87.92%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 292.64it/s]|
Column Pair Trends Score: 78.0%

Overall Score (Average): 82.96%



Loss: -1.427: 100%|███████████████████████████| 600/600 [00:24<00:00, 24.65it/s]


...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████████| 9/9 [00:00<00:00, 1033.08it/s]|
Column Shapes Score: 89.28%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 233.18it/s]|
Column Pair Trends Score: 70.71%

Overall Score (Average): 79.99%



Loss: -7.254: 100%|█████████████████████████| 1200/1200 [00:55<00:00, 21.48it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 9/9 [00:00<00:00, 917.66it/s]|
Column Shapes Score: 90.63%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 296.93it/s]|
Column Pair Trends Score: 78.12%

Overall Score (Average): 84.37%



Loss: -9.988: 100%|█████████████████████████| 1500/1500 [01:19<00:00, 18.95it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████████| 9/9 [00:00<00:00, 1034.92it/s]|
Column Shapes Score: 92.51%

(2/2) Evaluating Column Pair Trends: |████████| 36/36 [00:00<00:00, 322.12it/s]|
Column Pair Trends Score: 74.18%

Overall Score (Average): 83.34%

[0.786679616064274, 0.8295577025961575, 0.8366117277691862, 0.8418049328486923, 0.7999408477339135, 0.8437233251751028, 0.833449492543658]

Stability analysis — final window = 30% des epochs

 training_epochs  window_size epoch_range  Loss_slope
             300          180     210–299    -0.00826
             500          300     350–499    -0.00541
             600          360     420–599    -0.00348
             700          420     490–699    -0.00892
            1000          600     700–999    -0.00774
            1200          720    840–1199    -0.01253
            1500          900   1050–1499    -0.01392


#### AIDS dataset

In [198]:
# ============================================================
# Load PIMA dataset
# ============================================================
data_path = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data/AIDS_train.csv")

df = pd.read_csv(data_path,keep_default_na=False)

# ============================================================
# Detect metadata
# ============================================================

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name='hotel_guests'
)

# ============================================================
# CONFIGURATION
# ============================================================

output_dir = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data")
os.makedirs(output_dir, exist_ok=True)

# Training budgets
epoch_values = [1000,2000,3000,4000]


# ============================================================
# STORAGE
# ============================================================

loss_results = []
stability_results = []
quality_results =[]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

for epochs in epoch_values:
    synthesizer = TVAESynthesizer(
    metadata,
    epochs=epochs,               
    batch_size=256,
    compress_dims=(128,256),
    decompress_dims=(128, 256),
    l2scale=1e-3,
    verbose=True
)
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=len(df))
    # quality assessement
    quality_report = evaluate_quality(
        real_data=df,
        synthetic_data=synthetic_data,
        metadata=metadata
    )

    quality_score = (quality_report.get_score())
    quality_results.append(quality_score)

    # losses functions
    loss_df = (synthesizer.get_loss_values().copy())
    loss_df["training_epochs"] = epochs
    loss_results.append(loss_df)

aids_loss_df_all_tvae = pd.concat(loss_results,ignore_index=True)
aids_loss_df_all_tvae.to_csv(os.path.join(output_dir,"aids_loss_results_tvae.csv"),index=False)

print(quality_results)

if __name__ == "__main__":
    aids_summary_tvae = summarize_stability(aids_loss_df_all_tvae, window_fraction=final_window_fraction)

    pd.set_option("display.float_format", lambda x: f"{x:.5f}")
    print(f"\nStability analysis — final window = {int(final_window_fraction * 100)}% des epochs\n")
    print(aids_summary_tvae.to_string(index=False))
    aids_summary_tvae.to_csv(os.path.join(output_dir,"aids_slope_results_tvae.csv"),index=False)


Loss: -7.620: 100%|█████████████████████████| 1000/1000 [02:20<00:00,  7.11it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1305.17it/s]|
Column Shapes Score: 96.0%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 363.92it/s]|
Column Pair Trends Score: 93.9%

Overall Score (Average): 94.95%



Loss: -14.325: 100%|████████████████████████| 2000/2000 [04:33<00:00,  7.30it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 20/20 [00:00<00:00, 587.54it/s]|
Column Shapes Score: 96.19%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 412.27it/s]|
Column Pair Trends Score: 93.16%

Overall Score (Average): 94.67%



Loss: -16.462: 100%|████████████████████████| 3000/3000 [06:21<00:00,  7.86it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1290.77it/s]|
Column Shapes Score: 96.32%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 392.92it/s]|
Column Pair Trends Score: 93.38%

Overall Score (Average): 94.85%



Loss: -17.404: 100%|████████████████████████| 4000/4000 [08:01<00:00,  8.30it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |████████████| 20/20 [00:00<00:00, 1242.19it/s]|
Column Shapes Score: 96.5%

(2/2) Evaluating Column Pair Trends: |██████| 190/190 [00:00<00:00, 402.63it/s]|
Column Pair Trends Score: 93.25%

Overall Score (Average): 94.87%

[0.9495045646849254, 0.9467379202850144, 0.9485192607437097, 0.9487363616121132]

Stability analysis — final window = 30% des epochs

 training_epochs  window_size epoch_range  Loss_slope
            1000         1800     700–999    -0.01388
            2000         3600   1400–1999    -0.00485
            3000         5400   2100–2999    -0.00180
            4000         7200   2800–3999    -0.00125


#### REIN dataset

In [179]:
# ============================================================
# Load PIMA dataset
# ============================================================
data_path = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data/REIN_train.csv")

df = pd.read_csv(data_path,keep_default_na=False)

# ============================================================
# Detect metadata
# ============================================================

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name='hotel_guests'
)

# ============================================================
# CONFIGURATION
# ============================================================

output_dir = ("Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data")
os.makedirs(output_dir, exist_ok=True)

# Training budgets
epoch_values = [1000,2000,3000,4000]


# ============================================================
# STORAGE
# ============================================================

loss_results = []
stability_results = []
quality_results =[]

# ============================================================
# MAIN EXPERIMENT
# ============================================================

for epochs in epoch_values:
    synthesizer = TVAESynthesizer(
    metadata,
    epochs=epochs,               
    batch_size=500,      
    embedding_dim=64,
    compress_dims=(256,128),
    decompress_dims=(128, 256),
    l2scale=1e-5,
    #loss_factor=2,            
    #enforce_min_max_values=True,
    #enforce_rounding=True,
    #cuda=True,
    verbose=True
)
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=len(df))
    # quality assessement
    quality_report = evaluate_quality(
        real_data=df,
        synthetic_data=synthetic_data,
        metadata=metadata
    )

    quality_score = (quality_report.get_score())
    quality_results.append(quality_score)

    # losses functions
    loss_df = (synthesizer.get_loss_values().copy())
    loss_df["training_epochs"] = epochs
    loss_results.append(loss_df)

rein_loss_df_all_tvae = pd.concat(loss_results,ignore_index=True)
rein_loss_df_all_tvae.to_csv(os.path.join(output_dir,"rein_loss_results_tvae.csv"),index=False)

print(quality_results)

if __name__ == "__main__":
    rein_summary_tvae = summarize_stability(rein_loss_df_all_tvae, window_fraction=final_window_fraction)

    pd.set_option("display.float_format", lambda x: f"{x:.5f}")
    print(f"\nStability analysis — final window = {int(final_window_fraction * 100)}% des epochs\n")
    print(rein_summary_tvae.to_string(index=False))
    rein_summary_tvae.to_csv(os.path.join(output_dir,"rein_slope_results_tvae.csv"),index=False)


Loss: 1.924: 100%|██████████████████████████| 1000/1000 [23:13<00:00,  1.39s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 159.75it/s]|
Column Shapes Score: 92.87%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:00<00:00, 202.20it/s]|
Column Pair Trends Score: 95.96%

Overall Score (Average): 94.42%



Loss: 0.523: 100%|██████████████████████████| 2000/2000 [46:33<00:00,  1.40s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 161.47it/s]|
Column Shapes Score: 93.54%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:00<00:00, 202.62it/s]|
Column Pair Trends Score: 95.93%

Overall Score (Average): 94.74%



Loss: -0.365: 100%|███████████████████████| 3000/3000 [1:09:59<00:00,  1.40s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 139.39it/s]|
Column Shapes Score: 95.29%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:00<00:00, 206.97it/s]|
Column Pair Trends Score: 97.25%

Overall Score (Average): 96.27%



Loss: -1.449: 100%|███████████████████████| 4000/4000 [1:31:48<00:00,  1.38s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |█████████████| 19/19 [00:00<00:00, 147.58it/s]|
Column Shapes Score: 93.8%

(2/2) Evaluating Column Pair Trends: |██████| 171/171 [00:00<00:00, 203.39it/s]|
Column Pair Trends Score: 95.62%

Overall Score (Average): 94.71%

[0.9441639613717762, 0.9473728778811168, 0.9627302174754029, 0.9470827383467306]

Stability analysis — final window = 30% des epochs

 training_epochs  window_size epoch_range  Loss_slope
            1000        18000     700–999    -0.00310
            2000        36000   1400–1999    -0.00078
            3000        54000   2100–2999    -0.00069
            4000        72000   2800–3999    -0.00037


In [14]:
"""
============================================================
Training loss + synthetic data quality — TVAE
PIMA / AIDS / REIN
Publication-ready version
============================================================

Layout: 2 rows x 3 columns
    Row 1 : ELBO training loss (mean per epoch, PIMA | AIDS | REIN)
    Row 2 : Overall quality score vs. training epochs

The raw TVAE loss log records one value per (Epoch, Batch). Following
the requested procedure, the per-batch losses are first averaged
within each epoch, and all downstream analysis (curve, final-window
trend line) operates on this per-epoch mean loss.

Each dataset (column) uses its own number of training epochs,
matching the epochs used to train the TVAE synthesizer reported
in the paper for that dataset.
"""

import os
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ==================================================================
# PATHS
# ==================================================================

DATA_DIR = "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/data"
output_dir = "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/figures"
os.makedirs(output_dir, exist_ok=True)
# Training epochs retained for the reported TVAE model, per dataset
EPOCHS_CONFIG = {
    "PIMA": 600,
    "AIDS": 3000,
    "REIN": 2000,
}

FILES = {
    "PIMA": dict(
        loss=os.path.join(DATA_DIR, "pima_loss_results_tvae.csv"),
        slope=os.path.join(DATA_DIR, "pima_slope_results_tvae.csv"),
    ),
    "AIDS": dict(
        loss=os.path.join(DATA_DIR, "aids_loss_results_tvae.csv"),
        slope=os.path.join(DATA_DIR, "aids_slope_results_tvae.csv"),
    ),
    "REIN": dict(
        loss=os.path.join(DATA_DIR, "rein_loss_results_tvae.csv"),
        slope=os.path.join(DATA_DIR, "rein_slope_results_tvae.csv"),
    ),
}

QUALITY_FILE = os.path.join(DATA_DIR, "results_quality_tvae.csv")

# ==================================================================
# STYLE (publication-ready, colorblind-friendly, consistent with the
# CTGAN figure)
# ==================================================================

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 400,
    "font.family": "serif",
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "axes.labelsize": 11,
    "legend.fontsize": 9.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "lines.linewidth": 0.9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Colorblind-safe palette (Okabe-Ito)
COLOR_LOSS = "#0072B2"         # blue
COLOR_FIT_LOSS = "#08306b"     # dark blue
COLOR_WINDOW = "#7f7f7f"
COLOR_Q = "#009E73"            # bluish green
COLOR_Q_SELECTED = "#CC79A7"   # reddish purple

PANEL_LABELS = ["a", "b", "c", "d", "e", "f"]


# ==================================================================
# HELPERS
# ==================================================================

def _parse_epoch_range(epoch_range):
    parts = re.split(r"[\u2013\-]", str(epoch_range))
    if len(parts) != 2:
        raise ValueError(f"Unrecognized epoch range format: '{epoch_range}'")
    return int(parts[0]), int(parts[1])


def _fit_line_over_window(epochs, values, slope):
    intercept = np.mean(values) - slope * np.mean(epochs)
    return slope * epochs + intercept


def _average_loss_per_epoch(df):
    """
    Average the per-batch loss within each (training_epochs, Epoch) pair.

    The raw log contains one row per mini-batch (Epoch, Batch, Loss).
    This step must be applied before any trend/slope analysis, since the
    reported Loss_slope values were computed on the per-epoch mean loss.
    """
    required_cols = {"Epoch", "Batch", "Loss", "training_epochs"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in loss dataframe: {missing}")

    return (
        df.groupby(["training_epochs", "Epoch"], as_index=False)["Loss"]
        .mean()
        .sort_values(["training_epochs", "Epoch"])
        .reset_index(drop=True)
    )


def _prepare_dataset(name, df, slope_df, n_epochs):
    """Average per-batch losses per epoch, then extract the final window
    and its linear trend for one dataset."""
    epoch_mean_df = _average_loss_per_epoch(df)

    required_slope_cols = {"training_epochs", "epoch_range", "Loss_slope"}
    missing_slope = required_slope_cols - set(slope_df.columns)
    if missing_slope:
        raise ValueError(f"[{name}] Missing columns in slope dataframe: {missing_slope}")

    subset = (
        epoch_mean_df[epoch_mean_df["training_epochs"] == n_epochs]
        .sort_values("Epoch")
        .reset_index(drop=True)
    )
    if subset.empty:
        raise ValueError(f"[{name}] No data found for training_epochs = {n_epochs}.")

    slope_row = slope_df[slope_df["training_epochs"] == n_epochs].reset_index(drop=True)
    if slope_row.empty:
        raise ValueError(f"[{name}] No slope found for training_epochs = {n_epochs}.")

    row = slope_row.iloc[0]
    start, end = _parse_epoch_range(row["epoch_range"])

    window = subset[(subset["Epoch"] >= start) & (subset["Epoch"] <= end)]
    if window.empty:
        raise ValueError(f"[{name}] No data in the final window {start}-{end}.")

    window_epochs = window["Epoch"].to_numpy()
    loss_slope = float(row["Loss_slope"])
    fit_loss = _fit_line_over_window(window_epochs, window["Loss"].to_numpy(), loss_slope)

    return dict(
        name=name, n_epochs=n_epochs, subset=subset, window_epochs=window_epochs,
        fit_loss=fit_loss, start=start, end=end, loss_slope=loss_slope,
    )


def _plot_loss_panel(
    ax, subset, window_epochs, fit_values, start, end, slope,
    show_xlabel, show_title, title_text, show_ylabel, panel_label,
):
    ax.plot(subset["Epoch"], subset["Loss"], color=COLOR_LOSS, linewidth=0.9, alpha=0.9)
    ax.axvspan(start, end, color=COLOR_WINDOW, alpha=0.12, zorder=0)
    ax.plot(window_epochs, fit_values, color=COLOR_FIT_LOSS, linewidth=2.2,
            solid_capstyle="round")

    ax.set_xlim(subset["Epoch"].min(), subset["Epoch"].max())

    if show_xlabel:
        ax.set_xlabel("Epoch")
    if show_ylabel:
        ax.set_ylabel("Training loss")
    if show_title:
        ax.set_title(title_text)

    ax.text(
        -0.14, 1.08, panel_label, transform=ax.transAxes,
        fontsize=12, fontweight="bold", va="top", ha="left",
    )
    ax.text(
        0.97, 0.93,
        rf"slope $\approx$ {slope:+.1e}/epoch" + f"\n(epochs {start}\u2013{end})",
        transform=ax.transAxes, fontsize=7.8, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.28", facecolor="white",
                  edgecolor="0.6", alpha=0.85, linewidth=0.6),
    )


def _plot_quality_panel(
    ax, quality_subset, selected_epoch, show_xlabel, show_ylabel,
    panel_label, y_lim=None,
):
    epochs = quality_subset["epochs"].to_numpy()
    scores = quality_subset["overall_score"].to_numpy()

    ax.plot(epochs, scores, color=COLOR_Q, linewidth=1.3, marker="o",
            markersize=5, markerfacecolor="white", markeredgewidth=1.3, zorder=2)

    sel_row = quality_subset[quality_subset["epochs"] == selected_epoch]
    if not sel_row.empty:
        sel_score = float(sel_row["overall_score"].iloc[0])
        ax.plot(selected_epoch, sel_score, marker="*", markersize=15,
                color=COLOR_Q_SELECTED, markeredgecolor="0.2",
                markeredgewidth=0.6, zorder=3)
        ax.axvline(selected_epoch, color=COLOR_Q_SELECTED, linestyle=":",
                   linewidth=1.1, alpha=0.7, zorder=1)
        ax.annotate(
            f"{sel_score:.3f}",
            xy=(selected_epoch, sel_score),
            xytext=(0, 10), textcoords="offset points",
            fontsize=8.5, ha="center", fontweight="bold",
            color=COLOR_Q_SELECTED,
        )

    if y_lim is not None:
        ax.set_ylim(*y_lim)

    if show_xlabel:
        ax.set_xlabel("Training epochs")
    if show_ylabel:
        ax.set_ylabel("Overall quality score")

    ax.text(
        -0.14, 1.08, panel_label, transform=ax.transAxes,
        fontsize=12, fontweight="bold", va="top", ha="left",
    )


def plot_tvae_grid(datasets, output_dir, quality_df=None):
    n_datasets = len(datasets)
    prepared = [
        _prepare_dataset(d["name"], d["df"], d["slope_df"], d["n_epochs"])
        for d in datasets
    ]

    n_rows = 2 if quality_df is not None else 1
    fig, axes = plt.subplots(n_rows, n_datasets, figsize=(4.3 * n_datasets, 3.3 * n_rows), squeeze=False)

    for j, data in enumerate(prepared):
        is_first_col = j == 0
        column_title = f"{data['name']} ({data['n_epochs']} epochs)"

        _plot_loss_panel(
            ax=axes[0, j], subset=data["subset"], window_epochs=data["window_epochs"],
            fit_values=data["fit_loss"], start=data["start"], end=data["end"],
            slope=data["loss_slope"], show_xlabel=(quality_df is None),
            show_title=True, title_text=column_title,
            show_ylabel=is_first_col, panel_label=PANEL_LABELS[j],
        )

    if quality_df is not None:
        all_scores = quality_df["overall_score"].to_numpy()
        pad = 0.03 * (all_scores.max() - all_scores.min())
        shared_ylim = (all_scores.min() - pad, all_scores.max() + pad)

        for j, data in enumerate(prepared):
            name = data["name"]
            subset_q = (
                quality_df[quality_df["dataset"] == name]
                .sort_values("epochs")
                .reset_index(drop=True)
            )
            _plot_quality_panel(
                ax=axes[1, j],
                quality_subset=subset_q,
                selected_epoch=data["n_epochs"],
                show_xlabel=True,
                show_ylabel=(j == 0),
                panel_label=PANEL_LABELS[n_datasets + j],
                y_lim=shared_ylim,
            )

    legend_handles = [
        Line2D([0], [0], color=COLOR_LOSS, lw=1.4, label="Training loss"),
        Patch(facecolor=COLOR_WINDOW, alpha=0.25, label="Final analysis window"),
        Line2D([0], [0], color=COLOR_FIT_LOSS, lw=2.0, label="Linear fit (reported slope)"),
    ]
    if quality_df is not None:
        legend_handles += [
            Line2D([0], [0], color=COLOR_Q, lw=1.3, marker="o", markerfacecolor="white",
                   markersize=6, label="Overall quality score"),
            Line2D([0], [0], color=COLOR_Q_SELECTED, lw=0, marker="*", markersize=13,
                   label="Retained training epoch"),
        ]
    fig.legend(
        handles=legend_handles, loc="lower center", ncol=3,
        frameon=False, bbox_to_anchor=(0.5, -0.02),
    )

    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.subplots_adjust(wspace=0.30, hspace=0.38)

    png_path = os.path.join(output_dir, "tvae_training_losses_pima_aids_rein.png")
    pdf_path = os.path.join(output_dir, "tvae_training_losses_pima_aids_rein.pdf")

    fig.savefig(png_path, dpi=400, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    return png_path, pdf_path


def main():
    datasets = []
    for name, n_epochs in EPOCHS_CONFIG.items():
        loss_df = pd.read_csv(FILES[name]["loss"])
        slope_df = pd.read_csv(FILES[name]["slope"])
        datasets.append(dict(name=name, df=loss_df, slope_df=slope_df, n_epochs=n_epochs))

    quality_df = pd.read_csv(QUALITY_FILE) if os.path.exists(QUALITY_FILE) else None

    png_path, pdf_path = plot_tvae_grid(
        datasets=datasets, output_dir=output_dir, quality_df=quality_df,
    )
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


if __name__ == "__main__":
    main()

Saved: Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/figures/tvae_training_losses_pima_aids_rein.png
Saved: Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION/figures/tvae_training_losses_pima_aids_rein.pdf
